# Capstone Part 2: RFM Behavioral Segmentation & Risk Integration
**Course**: IITP AI Course Capstone
**Project Repository**: `d2c-churn-rfm-segmentation`

### Objective:
Engineers robust RFM features using transactional profiles on or before the 2025-09-30 snapshot limit, blends behavioral risk flags, assigns segments, and exports data files.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Part 2 dependencies loaded cleanly.')

Part 2 dependencies loaded cleanly.


## 1. Load Data and Isolate Pre-Snapshot Transactions
We enforce strict leakage boundaries by filtering out records past 2025-09-30.

In [2]:
orders = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')
support = pd.read_csv('support_tickets.csv')

orders_cleaned = orders[~orders['order_id'].str.endswith('_DUP')].copy()
orders_cleaned['order_date'] = pd.to_datetime(orders_cleaned['order_date'])
snapshot_date = pd.to_datetime('2025-09-30')

hist_orders = orders_cleaned[orders_cleaned['order_date'] <= snapshot_date].copy()
print(f'Safe transactions for modeling: {hist_orders.shape[0]}')

Safe transactions for modeling: 8128


## 2. RFM Feature Engineering
We compute core customer metrics: Recency, Frequency, and Monetary values.

In [3]:
rfm = hist_orders.groupby('customer_id').agg(
    last_date=('order_date', 'max'),
    frequency=('order_id', 'count'),
    monetary=('gross_amount', 'sum')
).reset_index()

rfm['recency'] = (snapshot_date - rfm['last_date']).dt.days
rfm = rfm.drop(columns=['last_date'])
print(rfm.head())

  customer_id  frequency  monetary  recency
0   CUST00001          6   2955.57      107
1   CUST00002          1    581.00       40
2   CUST00003          1    649.98      171
3   CUST00004          1   1604.04      131
4   CUST00005          4   2550.91       38


## 3. Integrating Non-RFM Operational Risk Signals
We supplement our profiles with support complaints and return rates per customer.

In [4]:
ticket_counts = support[pd.to_datetime(support['ticket_date']) <= snapshot_date].groupby('customer_id').size().reset_index(name='ticket_count')
return_metrics = hist_orders.groupby('customer_id')['returned'].mean().reset_index(name='return_rate')

master_df = pd.merge(customers[['customer_id']], rfm, on='customer_id', how='left').fillna({'recency': 999, 'frequency': 0, 'monetary': 0})
master_df = pd.merge(master_df, ticket_counts, on='customer_id', how='left').fillna({'ticket_count': 0})
master_df = pd.merge(master_df, return_metrics, on='customer_id', how='left').fillna({'return_rate': 0.0})
print(master_df.describe())

         frequency      monetary      recency  ticket_count  return_rate
count  2400.000000   2400.000000  2400.000000   2400.000000  2400.000000
mean      3.386667   2547.116829    87.375833      0.800417     0.067793
std       2.380725   2127.686276    80.137473      0.976890     0.182082
min       1.000000    149.000000     0.000000      0.000000     0.000000
25%       1.000000    954.525000    25.000000      0.000000     0.000000
50%       3.000000   2010.710000    66.000000      1.000000     0.000000
75%       5.000000   3562.115000   129.000000      1.000000     0.000000
max      16.000000  27215.920000   562.000000      6.000000     1.000000


## 4. Cohort Segmentation Definition
We define 5 distinct behavioral blocks using custom tier limits.

In [5]:
def segment_customer(row):
    if row['frequency'] == 0:
        return 'Dormant Unconverted'
    if row['recency'] <= 45 and row['frequency'] >= 5:
        return 'Champions'
    if row['recency'] > 90 and row['monetary'] > 2000:
        return 'At-Risk High-Value'
    if row['ticket_count'] >= 2 and row['return_rate'] > 0.3:
        return 'High-Value Unhappy'
    if row['recency'] <= 60 and row['frequency'] <= 2:
        return 'New Shoppers'
    return 'Regular Value Core'

master_df['segment_name'] = master_df.apply(segment_customer, axis=1)
print(master_df['segment_name'].value_counts())

master_df.to_csv('segments.csv', index=False)
print('segments.csv generated successfully.')

segment_name
Regular Value Core    1118
New Shoppers           515
At-Risk High-Value     428
Champions              292
High-Value Unhappy      47
Name: count, dtype: int64
segments.csv generated successfully.
